# Mini-CLIP Training
GPU runtime required. Runtime → Change runtime type → T4 GPU.

In [ ]:
# 1. mount drive and cd into repo
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML RESEARCH/mini-clip')
!pwd

In [ ]:
# 2. install deps
!pip install -q torch torchvision transformers wandb tqdm

In [ ]:
# 3. verify GPU
import torch
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 4. wandb login
import wandb
wandb.login()  # paste API key when prompted

In [ ]:
# 5. colab config overrides
from src.config import Config

cfg = Config(
    batch_size=256,
    device='cuda',
    num_workers=2,
    epochs=15,
)

In [ ]:
# 6. train
from src.train import train
train(cfg)

In [ ]:
# 7. eval on test set
from pathlib import Path
from src.model import MiniCLIP
from src.data import build_dataloaders
from src.eval import encode_all, recall_at_k, print_recalls
import torch

device = torch.device('cuda')
model = MiniCLIP().to(device)
model.load_state_dict(torch.load(f'{cfg.checkpoint_dir}/best.pt', map_location=device))

_, _, test_loader = build_dataloaders(
    captions_csv=Path(cfg.captions_csv),
    images_dir=Path(cfg.data_dir) / 'Images',
    splits_dir=Path(cfg.splits_dir),
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

img_embs, txt_embs = encode_all(model, test_loader, device)
results = recall_at_k(img_embs, txt_embs)
print_recalls(results)